# Phase 3 — Per-Well Stories

For each well: two-panel time-series (gas rate + WGR), one-paragraph story.
Plus cohort Gantt chart and breakthrough detection using WGR > 5 sustained ≥ 3 months.

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
elif 'mari_poc' not in os.getcwd(): os.chdir('mari_poc')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from src.config import (PROCESSED_DIR, FIGURES_DIR, COLOR_WET, COLOR_DRY,
                         COLOR_FORECAST, COLOR_EXCLUDED, HORIZONTAL_WELLS,
                         FORECAST_TARGETS, EXCLUDED_WELLS, WGR_THRESHOLD_BBL_MMCF)
from src.features import compute_wgr, detect_breakthrough

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_parquet(PROCESSED_DIR / 'panel_long.parquet')
static = pd.read_parquet(PROCESSED_DIR / 'well_static.parquet')

# Compute WGR
panel = compute_wgr(panel)

# Detect breakthrough
bt_df = detect_breakthrough(panel)
print('Breakthrough detection results:')
print(bt_df.to_string())

# Save breakthrough results
bt_df.to_parquet(PROCESSED_DIR / 'breakthrough_events.parquet', index=False)
print(f'\nSaved breakthrough_events.parquet')

Breakthrough detection results:
          well  bt_detected    bt_date bt_month_index
0     M-11-HRL         True 2012-04-01            409
1   M-122H-HRL         True 2023-12-01             12
2   M-123H-HRL        False        NaT           <NA>
3   M-124H-HRL        False        NaT           <NA>
4   M-125H-HRL        False        NaT           <NA>
5   M-126H-HRL        False        NaT           <NA>
6     M-13-HRL        False        NaT           <NA>
7     M-22-HRL        False        NaT           <NA>
8     M-41-HRL         True 2013-09-01            329
9     M-50-HRL         True 2017-11-01            379
10    M-51-HRL         True 2002-05-01            187
11    M-56-HRL         True 2018-06-01            299
12    M-57-HRL        False        NaT           <NA>
13    M-58-HRL         True 2000-11-01             88
14    M-61-HRL         True 2017-08-01            284
15    M-63-HRL         True 2016-12-01            275
16    M-65-HRL         True 2018-03-01            

In [2]:
# Per-well time-series plots
wells = sorted(panel['well'].unique())
n_wells = len(wells)
fig, axes = plt.subplots(n_wells, 1, figsize=(16, 4 * n_wells), sharex=False)

for i, well in enumerate(wells):
    ax1 = axes[i]
    grp = panel[panel['well'] == well].sort_values('date')
    
    # Gas rate (green)
    ax1.plot(grp['date'], grp['gas_mmcf'], color='green', alpha=0.7, linewidth=0.8)
    ax1.fill_between(grp['date'], 0, grp['gas_mmcf'], color='green', alpha=0.15)
    ax1.set_ylabel('Gas (MMcf/mo)', color='green', fontsize=8)
    ax1.tick_params(axis='y', labelcolor='green', labelsize=7)
    ax1.set_title(well, fontsize=10, fontweight='bold')
    
    # WGR on twin axis (blue)
    ax2 = ax1.twinx()
    wgr = grp['wgr_bbl_mmcf'].copy()
    wgr_clipped = wgr.clip(upper=100)  # clip for visibility
    ax2.plot(grp['date'], wgr_clipped, color='blue', alpha=0.6, linewidth=0.8)
    ax2.set_ylabel('WGR (bbl/MMcf)', color='blue', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='blue', labelsize=7)
    
    # Threshold line
    ax2.axhline(y=WGR_THRESHOLD_BBL_MMCF, color='red', linestyle='--', alpha=0.5, linewidth=0.7)
    
    # Breakthrough vertical line
    bt_row = bt_df[bt_df['well'] == well]
    if len(bt_row) > 0 and bt_row.iloc[0]['bt_detected']:
        bt_date = bt_row.iloc[0]['bt_date']
        ax1.axvline(x=bt_date, color='red', linestyle='-', alpha=0.7, linewidth=1.2)
        ax1.text(bt_date, ax1.get_ylim()[1]*0.9, f'BT', color='red', fontsize=7, ha='right')
    
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax1.tick_params(axis='x', labelsize=7)
    ax1.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_per_well_timeseries.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved 03_per_well_timeseries.png')

Saved 03_per_well_timeseries.png


In [3]:
# Cohort Gantt chart
fig, ax = plt.subplots(figsize=(16, 8))

# Sort wells by first_prod_date
static_sorted = static.sort_values('first_prod_date')
bt_lookup = bt_df.set_index('well')

for i, (_, row) in enumerate(static_sorted.iterrows()):
    well = row['well']
    start = row['first_prod_date']
    end = row['last_prod_date']
    
    # Color by category
    if well in EXCLUDED_WELLS:
        color = COLOR_EXCLUDED
        label = 'Excluded'
    elif well in FORECAST_TARGETS:
        color = COLOR_FORECAST
        label = 'Forecast target'
    elif well in bt_lookup.index and bt_lookup.loc[well, 'bt_detected']:
        color = COLOR_WET
        label = 'Wet'
    else:
        color = COLOR_DRY
        label = 'Dry'
    
    ax.barh(i, (end - start).days, left=start, height=0.6, color=color, alpha=0.8, edgecolor='white')
    ax.text(start - pd.Timedelta(days=200), i, well.replace('M-','').replace('-HRL',''),
            fontsize=7, ha='right', va='center')
    
    # Mark breakthrough
    if well in bt_lookup.index and bt_lookup.loc[well, 'bt_detected']:
        bt_d = bt_lookup.loc[well, 'bt_date']
        ax.plot(bt_d, i, 'rv', markersize=6)

ax.set_xlabel('Date')
ax.set_yticks([])
ax.set_title('Production Gantt Chart — All 22 Wells')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(axis='x', alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLOR_WET, label='Wet (breakthrough)'),
    Patch(facecolor=COLOR_DRY, label='Dry'),
    Patch(facecolor=COLOR_FORECAST, label='Forecast target'),
    Patch(facecolor=COLOR_EXCLUDED, label='Excluded'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_cohort_gantt.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 03_cohort_gantt.png')

Saved 03_cohort_gantt.png


## Per-Well Stories

In [4]:
# Generate per-well stories
bt_lookup = bt_df.set_index('well')
first_dates = static.set_index('well')['first_prod_date']

stories = {}
for well in sorted(panel['well'].unique()):
    grp = panel[panel['well'] == well].sort_values('date')
    gas = grp.dropna(subset=['gas_mmcf'])
    
    # Basic stats
    total_months = len(gas)
    peak_gas = gas['gas_mmcf'].max()
    peak_date = gas.loc[gas['gas_mmcf'].idxmax(), 'date'] if len(gas) > 0 else None
    first = first_dates.get(well)
    
    # Water info
    water_months = (grp['water_bbl'] > 0).sum()
    total_water = grp['water_bbl'].sum()
    
    # Breakthrough
    bt_det = bt_lookup.loc[well, 'bt_detected'] if well in bt_lookup.index else False
    bt_date = bt_lookup.loc[well, 'bt_date'] if (well in bt_lookup.index and bt_det) else None
    bt_month = bt_lookup.loc[well, 'bt_month_index'] if (well in bt_lookup.index and bt_det) else None
    
    # Current rate (last 3 months avg)
    last3 = gas.tail(3)['gas_mmcf'].mean() if len(gas) >= 3 else gas['gas_mmcf'].mean() if len(gas) > 0 else 0
    
    # Decline character
    if len(gas) > 24:
        early = gas.head(12)['gas_mmcf'].mean()
        recent = gas.tail(12)['gas_mmcf'].mean()
        decline_pct = (1 - recent/early) * 100 if early > 0 else 0
    else:
        decline_pct = None
    
    is_hz = well in HORIZONTAL_WELLS
    is_excl = well in EXCLUDED_WELLS
    is_target = well in FORECAST_TARGETS
    
    # Build story
    story = f"**{well}** — "
    if is_excl:
        story += f"EXCLUDED. Vertical well online {first:%Y-%m}. Produces water from month 0 (completion flowback). "
        story += f"{total_months} months of data, peak gas {peak_gas:.0f} MMcf/mo. "
        story += "Water production since day one makes breakthrough detection unreliable; excluded from survival analysis."
    elif is_target:
        story += f"FORECAST TARGET. Horizontal well online {first:%Y-%m}. "
        story += f"{total_months} months of data so far, current rate ~{last3:.0f} MMcf/mo. "
        story += f"Total water produced: {total_water:.0f} bbl. Currently dry — needs breakthrough prediction."
    elif bt_det:
        story += f"WET. {'Horizontal' if is_hz else 'Vertical'} well online {first:%Y-%m}. "
        story += f"Breakthrough at month {bt_month} ({bt_date:%Y-%m}). "
        story += f"Peak gas {peak_gas:.0f} MMcf/mo. "
        if decline_pct is not None:
            story += f"Overall decline ~{decline_pct:.0f}% from first year to last year. "
        if is_hz:
            story += "Only horizontal with observed breakthrough — key validation anchor."
    else:
        story += f"DRY. {'Horizontal' if is_hz else 'Vertical'} well online {first:%Y-%m}. "
        story += f"{total_months} months producing, peak gas {peak_gas:.0f} MMcf/mo. "
        if decline_pct is not None:
            story += f"Decline ~{decline_pct:.0f}% first-to-last year. "
        story += f"Water months: {water_months}/{total_months}. "
        if water_months > 0:
            story += f"Some water production but never sustained above WGR threshold."
        else:
            story += "No water production observed."
    
    stories[well] = story
    print(story)
    print()

**M-11-HRL** — WET. Vertical well online 1978-03. Breakthrough at month 409 (2012-04). Peak gas 322 MMcf/mo. Overall decline ~52% from first year to last year. 

**M-122H-HRL** — WET. Horizontal well online 2022-12. Breakthrough at month 12 (2023-12). Peak gas 409 MMcf/mo. Overall decline ~22% from first year to last year. Only horizontal with observed breakthrough — key validation anchor.

**M-123H-HRL** — FORECAST TARGET. Horizontal well online 2023-11. 20 months of data so far, current rate ~251 MMcf/mo. Total water produced: 0 bbl. Currently dry — needs breakthrough prediction.

**M-124H-HRL** — FORECAST TARGET. Horizontal well online 2023-12. 18 months of data so far, current rate ~428 MMcf/mo. Total water produced: 0 bbl. Currently dry — needs breakthrough prediction.

**M-125H-HRL** — FORECAST TARGET. Horizontal well online 2024-09. 10 months of data so far, current rate ~280 MMcf/mo. Total water produced: 0 bbl. Currently dry — needs breakthrough prediction.

**M-126H-HRL** — F

## Summary: 22-Well Characterizations

In [5]:
# One-sentence summary for each well
print('By the time we exit Phase 3, each of the 22 wells has a one-sentence characterization:\n')

for well in sorted(stories.keys()):
    bt_det = bt_lookup.loc[well, 'bt_detected'] if well in bt_lookup.index else False
    is_target = well in FORECAST_TARGETS
    is_excl = well in EXCLUDED_WELLS
    is_hz = well in HORIZONTAL_WELLS
    
    grp = panel[panel['well'] == well].sort_values('date')
    gas = grp.dropna(subset=['gas_mmcf'])
    first = first_dates.get(well)
    
    if is_excl:
        sentence = f"{well}: Excluded — water from month 0 (flowback artifact), unreliable for breakthrough analysis."
    elif is_target:
        sentence = f"{well}: Forecast target — horizontal well online {first:%Y-%m}, currently dry, needs P10/P50/P90 prediction."
    elif bt_det:
        bt_m = bt_lookup.loc[well, 'bt_month_index']
        sentence = f"{well}: Wet — breakthrough at month {bt_m}, {'horizontal' if is_hz else 'vertical'} well, validation anchor." if is_hz else f"{well}: Wet — breakthrough at month {bt_m}, vertical well."
    else:
        n_months = len(gas)
        sentence = f"{well}: Dry — {'horizontal' if is_hz else 'vertical'} well, {n_months} months producing, no sustained WGR > 5."
    
    print(sentence)

By the time we exit Phase 3, each of the 22 wells has a one-sentence characterization:

M-11-HRL: Wet — breakthrough at month 409, vertical well.
M-122H-HRL: Wet — breakthrough at month 12, horizontal well, validation anchor.
M-123H-HRL: Forecast target — horizontal well online 2023-11, currently dry, needs P10/P50/P90 prediction.
M-124H-HRL: Forecast target — horizontal well online 2023-12, currently dry, needs P10/P50/P90 prediction.
M-125H-HRL: Forecast target — horizontal well online 2024-09, currently dry, needs P10/P50/P90 prediction.
M-126H-HRL: Forecast target — horizontal well online 2024-10, currently dry, needs P10/P50/P90 prediction.
M-13-HRL: Dry — vertical well, 562 months producing, no sustained WGR > 5.
M-22-HRL: Dry — vertical well, 524 months producing, no sustained WGR > 5.
M-41-HRL: Wet — breakthrough at month 329, vertical well.
M-50-HRL: Wet — breakthrough at month 379, vertical well.
M-51-HRL: Excluded — water from month 0 (flowback artifact), unreliable for brea

## Findings

1. Breakthrough detection identifies wet wells using WGR > 5 bbl/MMcf sustained ≥ 3 consecutive months
2. M-122H is the only horizontal with breakthrough (month ~12) — key validation anchor
3. Several vertical wells show water production but never reach sustained WGR > 5
4. M-51 excluded — water from month 0 is flowback, not formation water breakthrough
5. Forecast targets (M-123H through M-126H) all currently dry with short production histories (9-20 months)
6. Oldest wells (M-11, M-13) have 45+ years of production — massive decline but still producing
7. Gas rate decline patterns vary: some wells show sharp initial decline then stabilization, others gradual
8. Water production in vertical wells tends to appear gradually, not as a sharp front
9. The 4 forecast targets have very limited production history — predictions will have high uncertainty
10. Horizontal wells have higher initial rates than verticals (as expected for longer completion intervals)